In [3]:
import rasterio
import numpy as np
from datetime import datetime

def choose_template(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    month = dt.month

    if month in [2, 3, 4]:
        return "/content/drive/MyDrive/GitHub/no2_prediction_pipeline/data/GEE_Surgut_NO2/Norm/Surgut_NO2_FebApr_template_normalized.tif"
    elif month in [5, 6, 7, 8]:
        return "/content/drive/MyDrive/GitHub/no2_prediction_pipeline/data/GEE_Surgut_NO2/Norm/Surgut_NO2_MayAug_template_normalized.tif"
    elif month in [9, 10, 11]:
        return "/content/drive/MyDrive/GitHub/no2_prediction_pipeline/data/GEE_Surgut_NO2/Norm/Surgut_NO2_SepNov_template_normalized.tif"
    else:
        raise ValueError("Дата вне рабочего периода февраль-ноябрь")

def make_daily_map(date_str: str, predicted_no2: float, output_tif: str):
    template_path = choose_template(date_str)

    with rasterio.open(template_path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()

    mask = np.isfinite(arr)

    daily_arr = np.full(arr.shape, np.nan, dtype="float32")
    daily_arr[mask] = arr[mask] * predicted_no2

    profile.update(dtype="float32")

    with rasterio.open(output_tif, "w", **profile) as dst:
        dst.write(daily_arr, 1)

    print(f"Использован шаблон: {template_path}")
    print(f"Дата: {date_str}")
    print(f"Прогноз модели: {predicted_no2}")
    print(f"Сохранено: {output_tif}")
    print("Min:", np.nanmin(daily_arr))
    print("Max:", np.nanmax(daily_arr))
    print("Mean:", np.nanmean(daily_arr))

In [4]:
make_daily_map(
    date_str="2024-03-15",
    predicted_no2=40.0,
    output_tif="Surgut_NO2_2024-03-15_predicted_map.tif"
)

Использован шаблон: /content/drive/MyDrive/GitHub/no2_prediction_pipeline/data/GEE_Surgut_NO2/Norm/Surgut_NO2_FebApr_template_normalized.tif
Дата: 2024-03-15
Прогноз модели: 40.0
Сохранено: Surgut_NO2_2024-03-15_predicted_map.tif
Min: 20.693605
Max: 96.91533
Mean: 40.0
